# Universal Policy Metrics & 30-Year Backtest Evaluation
This notebook provides a deep dive into the internal metrics of the Multi-Agent RL Transformer. It visualizes the latent **Asset Embeddings** learned by the model and evaluates the recommended portfolio against a 30-year historical baseline (S&P 500).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from sklearn.decomposition import PCA

# Add root and backend to path
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('./backend'))

from vector_encoder import RLRecommender
from _constants import TEST_PROFILES

plt.style.use('dark_background') # Premium aesthetic

print("Initializing RL Recommender Engine...")
recommender = RLRecommender()
print(f"Engine Ready: {recommender._initialized}")
print(f"Loaded {len(recommender.dynamic_embeddings)} dynamic asset embeddings.")
print(f"Loaded {len(recommender.models)} RL Ensemble Agents.")

### 1. Asset Embedding Visualization (PCA Projection)
The transformer learns a high-dimensional embedding for each asset based on its price history, volatility, and fundamental metrics. Here, we reduce these embeddings to 2D to see how the model "groups" similar assets together.

In [ ]:
# Extract embeddings
tickers = list(recommender.dynamic_embeddings.keys())
if not tickers:
    print("ERROR: No embeddings found. Please run the data sync / ML worker first.")
else:
    embeddings = np.array([recommender.dynamic_embeddings[t] for t in tickers])

    # Reduce to 2D using Principal Component Analysis
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    # Plot the entire universe
    plt.figure(figsize=(14, 10))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], alpha=0.3, c='#00E676', s=15, edgecolors='none')

    # Highlight and label a few recognizable major tickers
    highlight_tickers = ['AAPL', 'MSFT', 'TSLA', 'JNJ', 'JPM', 'NVDA', 'SPY', 'QQQ', 'TLT', 'GLD']
    for i, t in enumerate(tickers):
        if t in highlight_tickers:
            plt.scatter(emb_2d[i, 0], emb_2d[i, 1], c='#FF3D00', s=80, edgecolors='white', zorder=5)
            plt.annotate(t, (emb_2d[i, 0]+0.05, emb_2d[i, 1]+0.05), fontsize=12, weight='bold', color='white')

    plt.title("Asset Semantic Landscape (2D PCA of Learned Embeddings)", fontsize=16, weight='bold')
    plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)", fontsize=12)
    plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)", fontsize=12)
    plt.grid(alpha=0.15)
    plt.show()

### 1.1 Raw Embedding Inspection
Use the cell below to inspect the raw 8-dimensional latent vectors for any asset. You can either list specific tickers in `my_tickers` or set `show_all = True` to dump the entire universe (caution: very large).

In [ ]:
# ADJUST THESE SETTINGS
my_tickers = ['AAPL', 'MSFT', 'JNJ', 'SPY', 'BTC-USD', 'TSLA'] # Add any tickers you want here
show_all = False # Set to True to see all 6,000+ assets
limit = 50 # If show_all is True, limit the output to the first N assets

target_list = tickers if show_all else my_tickers
if show_all:
    target_list = target_list[:limit]
    print(f"Displaying first {limit} assets in the universe...\n")

print(f"{'Ticker':<10} | {'Raw Latent Vector (8-Dim)':<40}")
print("-"*65)

count = 0
for t in target_list:
    if t in recommender.dynamic_embeddings:
        vec = recommender.dynamic_embeddings[t]
        vec_str = ' '.join([f'{v:>7.3f}' for v in vec])
        print(f"{t:<10} | [{vec_str} ]")
        count += 1
    elif not show_all:
        print(f"{t:<10} | [ ERROR: Ticker not found in embedding cache ]")

if count == 0 and not show_all:
    print("\nNo valid tickers found. Please check your spelling or verify data sync.")

### 2. Multi-Goal Profile Configuration
We configure a set of goals with different horizons (e.g. Downpayment in 5 years, Retirement in 25 years). The RL model generates different weights for each "Phase" of the backtest.

In [ ]:
from vector_encoder import encode_multi_horizon

# Create a Multi-Goal request
multi_goal_answers = {
    "start_cap": 100000,
    "monthly_contrib": 1000,
    "drawdown_sensitivity": 4,
    "volatility_sensitivity": 4,
    "goals": [
        {"name": "House Downpayment", "amount": 200000, "years": 5},
        {"name": "Retirement", "amount": 2000000, "years": 25}
    ]
}

# ── Determine Point-in-Time anchor for 30-year backtest ──
years_to_test = 15
returns = recommender.daily_returns
days_to_test = min(years_to_test * 252, len(returns))
back_returns = returns.iloc[-days_to_test:]
backtest_start_date = str(back_returns.index[0].date())
print(f"Backtest Start Date (Point-in-Time Anchor): {backtest_start_date}")

# Request recommendations using ONLY assets that existed at the start of the simulation
print("Requesting Multi-Horizon weights with strict Point-in-Time filtering...")
res = encode_multi_horizon(
    multi_goal_answers,
    require_historical_years=years_to_test,
    as_of_date=backtest_start_date
)

# FIX: encode_multi_horizon returns a flat dictionary, not nested under 'portfolio'
segments = res.get('segments', [])

if not segments:
    print("\nWARNING: No segments were generated. Diagnostics:")
    print(f"  - Recommender Models Loaded: {len(recommender.models)}")
    print(f"  - Input Goals: {len(multi_goal_answers.get('goals', []))}")
    print(f"  - Full API Response Keys: {list(res.keys())}")
    if 'error' in res:
        print(f"  - ERROR MESSAGE: {res['error']}")
else:
    print(f"Successfully generated {len(segments)} segments.")
    for i, seg in enumerate(segments):
        print(f"\nPHASE {i+1}: {seg['goal_name']} (Years {seg['horizon_years'][0]} - {seg['horizon_years'][1]})")
        w = seg.get('weights', {})
        if not w:
            print("  -> No assets selected for this phase (weights are empty).")
        else:
            for t, val in sorted(w.items(), key=lambda x: x[1], reverse=True)[:10]:
                # Show IPO date to verify no look-ahead leakage
                first_date = 'N/A'
                if t in returns.columns:
                    valid = returns[t].dropna()
                    if len(valid) > 0:
                        first_date = str(valid.index[0].date())
                print(f"  {t:<8}: {val:>6.2%}  (First Data: {first_date})")


### 3. Multi-Phase 30-Year POT Backtest
This simulation switches the portfolio weights as it passes through the "Phase" boundaries (Target Date Fund logic) over a 30-year historical window.

In [ ]:
import matplotlib.ticker as ticker

if not segments:
    print("Skipping backtest because no segments were generated.")
else:
    returns = recommender.daily_returns
    years_to_test = 15
    days_to_test = years_to_test * 252

    if len(returns) < days_to_test:
        days_to_test = len(returns)

    back_returns = returns.iloc[-days_to_test:]
    dates = back_returns.index
    backtest_start_date = str(dates[0].date())

    # Get all unique assets across all phases
    all_assets = set()
    for seg in segments:
        all_assets.update(seg['weights'].keys())

    # ── STRICT FILTERING: Only allow assets that existed BEFORE the backtest start ──
    verified_assets = []
    rejected_assets = []
    for t in all_assets:
        if t not in back_returns.columns:
            rejected_assets.append((t, 'not in returns matrix'))
            continue
        # Check: does this asset have non-NaN returns on the FIRST day of the backtest?
        first_valid = back_returns[t].first_valid_index()
        if first_valid is None:
            rejected_assets.append((t, 'no valid data at all'))
            continue
        # Allow a small grace period (first 5 trading days) for data alignment
        if first_valid > dates[min(5, len(dates)-1)]:
            rejected_assets.append((t, f'IPO after start: first data on {first_valid.date()}'))
            continue
        verified_assets.append(t)

    if rejected_assets:
        print(f"\n⚠️  REJECTED {len(rejected_assets)} assets that did not exist at backtest start ({backtest_start_date}):")
        for t, reason in rejected_assets:
            print(f"    {t}: {reason}")

    all_assets = verified_assets
    print(f"\n✅ Using {len(all_assets)} verified assets that existed on {backtest_start_date}")
    if not all_assets:
        all_assets = ['SPY'] if 'SPY' in back_returns.columns else [back_returns.columns[0]]
        print(f"   Fallback to: {all_assets}")

    port_returns_matrix = back_returns[all_assets]

    # Recompute target weights using ONLY verified assets
    target_df = pd.DataFrame(0.0, index=dates, columns=all_assets)
    for i, dt in enumerate(dates):
        elapsed_years = (dt - dates[0]).days / 365.25
        # Match to segment based on elapsed time from start
        chosen_seg = segments[-1]
        for seg in segments:
            yr_start, yr_end = seg['horizon_years']
            if yr_start <= elapsed_years < yr_end:
                chosen_seg = seg
                break
        # Renormalize weights to only include verified assets
        seg_weights = {t: w for t, w in chosen_seg['weights'].items() if t in all_assets}
        total_w = sum(seg_weights.values())
        if total_w > 0:
            for t, w in seg_weights.items():
                target_df.loc[dt, t] = w / total_w

    # No need for active_mask anymore — all assets are verified to exist from day 1
    daily_weights = target_df
    missing_weight = 1.0 - daily_weights.sum(axis=1)

    spy_col = '^GSPC' if '^GSPC' in back_returns.columns else 'SPY' if 'SPY' in back_returns.columns else None
    if spy_col:
        spy_returns = back_returns[spy_col].fillna(0.0)
    else:
        spy_returns = back_returns.mean(axis=1)

    port_daily_rets = (port_returns_matrix.fillna(0.0) * daily_weights).sum(axis=1) + (spy_returns * missing_weight.clip(lower=0))

    # ── High-Fidelity Capital Path Simulation ──
    initial_capital = multi_goal_answers.get('start_cap', 100000)
    monthly_contrib = multi_goal_answers.get('monthly_contrib', 1000)
    goals_list = multi_goal_answers.get('goals', [])

    # Map goals to exact day indices based on years from START
    goal_withdrawals = {}
    for g in goals_list:
        y = g['years']
        target_idx = min(int(y * 252), len(dates) - 1)
        goal_withdrawals[target_idx] = (g['amount'], g['name'])

    # RL Portfolio Simulation
    rl_balance = initial_capital
    rl_path = []
    rl_withdrawn = 0.0
    rl_shortfall = 0.0

    # S&P 500 Simulation
    spy_balance = initial_capital
    spy_path = []
    spy_withdrawn = 0.0
    spy_shortfall = 0.0

    total_contributions = 0.0

    for idx, dt in enumerate(dates):
        # 1. Update balances with daily return
        rl_balance *= (1.0 + port_daily_rets.iloc[idx])
        spy_balance *= (1.0 + spy_returns.iloc[idx])

        # 2. Add monthly contributions (every 21 trading days)
        if idx > 0 and idx % 21 == 0:
            rl_balance += monthly_contrib
            spy_balance += monthly_contrib
            total_contributions += monthly_contrib

        # 3. Deduct financial goals
        if idx in goal_withdrawals:
            amt, name = goal_withdrawals[idx]

            # RL deduction
            if rl_balance >= amt:
                rl_balance -= amt
                rl_withdrawn += amt
            else:
                withdrawn = max(rl_balance, 0.0)
                rl_withdrawn += withdrawn
                rl_shortfall += (amt - withdrawn)
                rl_balance = 0.0

            # S&P 500 deduction
            if spy_balance >= amt:
                spy_balance -= amt
                spy_withdrawn += amt
            else:
                withdrawn = max(spy_balance, 0.0)
                spy_withdrawn += withdrawn
                spy_shortfall += (amt - withdrawn)
                spy_balance = 0.0

        rl_path.append(rl_balance)
        spy_path.append(spy_balance)

    rl_path = pd.Series(rl_path, index=dates)
    spy_path = pd.Series(spy_path, index=dates)

    # ── Performance Metrics ──
    rl_sharpe = (port_daily_rets.mean() / port_daily_rets.std()) * np.sqrt(252) if port_daily_rets.std() > 0 else 0
    spy_sharpe = (spy_returns.mean() / spy_returns.std()) * np.sqrt(252) if spy_returns.std() > 0 else 0

    rl_cum_pure = (1 + port_daily_rets).cumprod()
    rl_mdd = (rl_cum_pure / rl_cum_pure.cummax() - 1).min()
    spy_cum_pure = (1 + spy_returns).cumprod()
    spy_mdd = (spy_cum_pure / spy_cum_pure.cummax() - 1).min()

    # Annualized return
    n_years = len(dates) / 252
    rl_total_return = rl_cum_pure.iloc[-1]
    spy_total_return = spy_cum_pure.iloc[-1]
    rl_ann_ret = (rl_total_return ** (1 / n_years) - 1) if rl_total_return > 0 else 0
    spy_ann_ret = (spy_total_return ** (1 / n_years) - 1) if spy_total_return > 0 else 0

    # Annualized volatility
    rl_ann_vol = port_daily_rets.std() * np.sqrt(252)
    spy_ann_vol = spy_returns.std() * np.sqrt(252)

    # Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [3, 1]})

    # Top: Capital path
    ax1.plot(dates, rl_path, label='RL Multi-Goal Portfolio', color='#00E676', linewidth=2.5)
    ax1.plot(dates, spy_path, label='S&P 500 Baseline', color='#B0BEC5', linewidth=1.5, linestyle='--')
    for idx_g, (amt, name) in goal_withdrawals.items():
        dt_g = dates[idx_g]
        ax1.axvline(dt_g, color='#FF5252', alpha=0.6, linestyle=':', linewidth=1.5)
        ax1.annotate(f'{name}\n-${amt:,.0f}', xy=(dt_g, ax1.get_ylim()[1]*0.4), fontsize=9, color='#FF5252', rotation=90, ha='right')
    ax1.set_title('30-Year High-Fidelity Glide Path Backtest: RL vs S&P 500', fontsize=16, weight='bold', pad=15)
    ax1.set_ylabel('Portfolio Balance ($)', fontsize=12)
    ax1.set_yscale('log')
    ax1.grid(alpha=0.15)
    ax1.legend(loc='upper left', framealpha=0.8)
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: f'${y:,.0f}'))

    # Bottom: Rolling 1-year return comparison
    rolling_window = 252
    rl_rolling = port_daily_rets.rolling(rolling_window).apply(lambda x: (1+x).prod() - 1, raw=True)
    spy_rolling = spy_returns.rolling(rolling_window).apply(lambda x: (1+x).prod() - 1, raw=True)
    ax2.plot(dates, rl_rolling * 100, color='#00E676', linewidth=1.5, alpha=0.8, label='RL (1yr rolling)')
    ax2.plot(dates, spy_rolling * 100, color='#B0BEC5', linewidth=1.5, alpha=0.8, linestyle='--', label='S&P 500 (1yr rolling)')
    ax2.axhline(0, color='white', alpha=0.3, linewidth=0.5)
    ax2.set_ylabel('Rolling 1Y Return (%)', fontsize=11)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.legend(loc='upper left', fontsize=9)
    ax2.grid(alpha=0.15)

    plt.tight_layout()
    plt.show()

    # ── Print Comparative Metrics Table ──
    print("=" * 70)
    print(f"{'':>20} BACKTEST PERFORMANCE SUMMARY")
    print(f"{'':>20} {backtest_start_date} → {str(dates[-1].date())}")
    print("=" * 70)
    print(f"Initial Capital:           ${initial_capital:>15,.2f}")
    print(f"Total Contributions Added: ${total_contributions:>15,.2f}")
    print(f"Verified Assets Used:       {len(all_assets):>15d}")
    print("-" * 70)
    print(f"{'Metric':<30} | {'RL Portfolio':>18} | {'S&P 500':>18}")
    print("-" * 70)
    print(f"{'Final Balance':<30} | ${rl_path.iloc[-1]:>17,.2f} | ${spy_path.iloc[-1]:>17,.2f}")
    print(f"{'Total Goals Withdrawn':<30} | ${rl_withdrawn:>17,.2f} | ${spy_withdrawn:>17,.2f}")
    print(f"{'Goal Shortfall':<30} | ${rl_shortfall:>17,.2f} | ${spy_shortfall:>17,.2f}")
    print(f"{'Annualized Return (Pure)':<30} | {rl_ann_ret*100:>16.2f}% | {spy_ann_ret*100:>16.2f}%")
    print(f"{'Annualized Volatility':<30} | {rl_ann_vol*100:>16.2f}% | {spy_ann_vol*100:>16.2f}%")
    print(f"{'Annualized Sharpe Ratio':<30} | {rl_sharpe:>17.2f} | {spy_sharpe:>17.2f}")
    print(f"{'Max Drawdown (Pure)':<30} | {rl_mdd*100:>16.2f}% | {spy_mdd*100:>16.2f}%")
    print("=" * 70)

    # Top 10 holdings
    print(f"\n{'':>20} TOP 10 HOLDINGS BY PHASE")
    print("-" * 70)
    for i, seg in enumerate(segments):
        w = {t: v for t, v in seg.get('weights', {}).items() if t in all_assets}
        total_w = sum(w.values())
        if total_w > 0:
            w = {t: v/total_w for t, v in w.items()}
        print(f"\nPhase {i+1}: {seg['goal_name']} (Years {seg['horizon_years'][0]}-{seg['horizon_years'][1]})")
        for t, val in sorted(w.items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  {t:<8}: {val:>6.2%}")


### 4. Full TEST_PROFILES Comparative Analysis
This section evaluates **all 30 TEST_PROFILES** from `_constants.py` — the same profiles used to train the RL agents. Profiles are grouped into 3 risk categories (Conservative, Balanced, Aggressive) and backtested with full Point-in-Time constraints. We compare per-category averaged metrics, RL training penalties, and asset concentration to prove that user inputs systematically influence portfolio composition.

In [ ]:
import matplotlib.ticker as ticker
from _constants import CAGR_REWARD_SCALE, MDD_PENALTY_SCALE, VOL_PENALTY_SCALE, DIVERSITY_PENALTY_SCALE, TEST_PROFILES
from vector_encoder import encode_multi_horizon

# ── Group TEST_PROFILES into risk categories ──
conservative_profiles = TEST_PROFILES[:10]  # First 10: Conservative
balanced_profiles     = TEST_PROFILES[10:20] # Next 10: Balanced
aggressive_profiles   = TEST_PROFILES[20:]   # Last 10: Aggressive

categories = {
    'Conservative': conservative_profiles,
    'Balanced':     balanced_profiles,
    'Aggressive':   aggressive_profiles,
}

years_to_test = 15
returns = recommender.daily_returns
days_to_test = min(years_to_test * 252, len(returns))
back_returns = returns.iloc[-days_to_test:]
dates = back_returns.index
backtest_start_date = str(dates[0].date())
n_years = len(dates) / 252

print(f'Backtest Window: {backtest_start_date} → {str(dates[-1].date())} ({n_years:.1f} years)')
print(f'Total TEST_PROFILES: {len(TEST_PROFILES)} ({len(conservative_profiles)} Cons / {len(balanced_profiles)} Bal / {len(aggressive_profiles)} Agg)')
print('=' * 80)

# ── Run all 30 profiles ──
all_profile_results = []

for cat_name, profiles in categories.items():
    print(f'\n── {cat_name} ({len(profiles)} profiles) ──')
    for p in profiles:
        p_name = p['profile_name']
        
        # Convert goals dict format to list format for encode_multi_horizon
        goals_list = [{'name': f'Goal Y{y}', 'amount': float(a), 'years': int(y)} for y, a in p['goals'].items()]
        
        answers = {
            'start_cap': p['start_cap'],
            'monthly_contrib': p.get('monthly_contrib', 0),
            'drawdown_sensitivity': p['drawdown_sensitivity'],
            'volatility_sensitivity': p['volatility_sensitivity'],
            'goal_flexibility': p.get('goal_flexibility', 5),
            'concentration_pref': p.get('concentration_pref', 5),
            'goals': goals_list,
        }
        
        res = encode_multi_horizon(answers, require_historical_years=years_to_test, as_of_date=backtest_start_date)
        segments = res.get('segments', [])
        
        if not segments:
            print(f'  ⚠️ {p_name}: No segments generated, skipping.')
            continue
        
        # Collect and verify assets
        all_assets = set()
        for seg in segments:
            all_assets.update(seg['weights'].keys())
        
        verified_assets = []
        for t in all_assets:
            if t in back_returns.columns:
                fv = back_returns[t].first_valid_index()
                if fv is not None and fv <= dates[min(5, len(dates)-1)]:
                    verified_assets.append(t)
        
        if not verified_assets:
            print(f'  ⚠️ {p_name}: No verified assets, skipping.')
            continue
        
        # Daily weights
        target_df = pd.DataFrame(0.0, index=dates, columns=verified_assets)
        for dt in dates:
            elapsed_years = (dt - dates[0]).days / 365.25
            chosen_seg = segments[-1]
            for seg in segments:
                yr_start, yr_end = seg['horizon_years']
                if yr_start <= elapsed_years < yr_end:
                    chosen_seg = seg
                    break
            seg_w = {t: w for t, w in chosen_seg['weights'].items() if t in verified_assets}
            tw = sum(seg_w.values())
            if tw > 0:
                for t, w in seg_w.items():
                    target_df.loc[dt, t] = w / tw
        
        port_returns_matrix = back_returns[verified_assets]
        port_daily_rets = (port_returns_matrix.fillna(0.0) * target_df).sum(axis=1)
        
        # Pure metrics (no withdrawals)
        cum = (1 + port_daily_rets).cumprod()
        mdd = (cum / cum.cummax() - 1).min()
        sharpe = (port_daily_rets.mean() / port_daily_rets.std()) * np.sqrt(252) if port_daily_rets.std() > 0 else 0
        ann_vol = port_daily_rets.std() * np.sqrt(252)
        total_return = cum.iloc[-1]
        ann_ret = (total_return ** (1 / n_years) - 1) if total_return > 0 else 0
        
        # RL Training Penalties
        dd_sens = p['drawdown_sensitivity']
        vol_sens = p['volatility_sensitivity']
        conc_pref = p.get('concentration_pref', 5)
        mdd_penalty = abs(mdd) * MDD_PENALTY_SCALE * dd_sens
        vol_penalty = ann_vol * VOL_PENALTY_SCALE * vol_sens
        div_threshold = max(3, int(11 - conc_pref))
        div_penalty = max(0, len(verified_assets) - div_threshold) * DIVERSITY_PENALTY_SCALE
        
        # Top 5 holdings (last segment = longest horizon)
        last_seg_w = {t: v for t, v in segments[-1].get('weights', {}).items() if t in verified_assets}
        tw = sum(last_seg_w.values())
        if tw > 0:
            last_seg_w = {t: v/tw for t, v in last_seg_w.items()}
        top5 = sorted(last_seg_w.items(), key=lambda x: x[1], reverse=True)[:5]
        top1_weight = top5[0][1] if top5 else 0
        top5_weight = sum(v for _, v in top5)
        
        result = {
            'name': p_name,
            'category': cat_name,
            'dd_sens': dd_sens,
            'vol_sens': vol_sens,
            'conc_pref': conc_pref,
            'ann_ret': ann_ret,
            'ann_vol': ann_vol,
            'sharpe': sharpe,
            'mdd': mdd,
            'mdd_penalty': mdd_penalty,
            'vol_penalty': vol_penalty,
            'div_penalty': div_penalty,
            'n_assets': len(verified_assets),
            'top1_weight': top1_weight,
            'top5_weight': top5_weight,
            'top5': top5,
            'daily_rets': port_daily_rets,
        }
        all_profile_results.append(result)
        print(f'  ✅ {p_name:30s} | Return: {ann_ret*100:5.1f}% | Vol: {ann_vol*100:5.1f}% | MDD: {mdd*100:6.1f}% | Sharpe: {sharpe:.2f} | Assets: {len(verified_assets)}')

print(f'\n\nCompleted {len(all_profile_results)} / {len(TEST_PROFILES)} profiles successfully.')


### 5. Category-Averaged Metrics & Penalty Analysis
Aggregate the per-profile metrics into category averages to show the systematic impact of user sensitivity parameters on portfolio construction.

In [ ]:
# ── S&P 500 Baseline (Pure, no withdrawals) ──
spy_col = '^GSPC' if '^GSPC' in back_returns.columns else 'SPY' if 'SPY' in back_returns.columns else None
spy_returns = back_returns[spy_col].fillna(0.0) if spy_col else back_returns.mean(axis=1)
spy_cum = (1 + spy_returns).cumprod()
spy_mdd = (spy_cum / spy_cum.cummax() - 1).min()
spy_sharpe = (spy_returns.mean() / spy_returns.std()) * np.sqrt(252) if spy_returns.std() > 0 else 0
spy_ann_vol = spy_returns.std() * np.sqrt(252)
spy_ann_ret = (spy_cum.iloc[-1] ** (1 / n_years) - 1)

# ── Build category summary table ──
results_df = pd.DataFrame(all_profile_results)

cat_summary = results_df.groupby('category').agg(
    profiles=('name', 'count'),
    avg_dd_sens=('dd_sens', 'mean'),
    avg_vol_sens=('vol_sens', 'mean'),
    avg_conc_pref=('conc_pref', 'mean'),
    avg_return=('ann_ret', 'mean'),
    avg_vol=('ann_vol', 'mean'),
    avg_sharpe=('sharpe', 'mean'),
    avg_mdd=('mdd', 'mean'),
    avg_mdd_penalty=('mdd_penalty', 'mean'),
    avg_vol_penalty=('vol_penalty', 'mean'),
    avg_div_penalty=('div_penalty', 'mean'),
    avg_n_assets=('n_assets', 'mean'),
    avg_top1_wt=('top1_weight', 'mean'),
    avg_top5_wt=('top5_weight', 'mean'),
).reindex(['Conservative', 'Balanced', 'Aggressive'])

# ── Print the Master Comparison Table ──
print('=' * 110)
print(f"{'':>35} CATEGORY-AVERAGED BACKTEST METRICS")
print(f"{'':>35} {backtest_start_date} → {str(dates[-1].date())} ({n_years:.1f} years)")
print('=' * 110)
print(f"{'Metric':<30} | {'Conservative':>14} | {'Balanced':>14} | {'Aggressive':>14} | {'S&P 500':>14}")
print('-' * 110)

# Input Parameters
print(f"{'Avg Drawdown Sensitivity':<30} | {cat_summary.loc['Conservative','avg_dd_sens']:>14.1f} | {cat_summary.loc['Balanced','avg_dd_sens']:>14.1f} | {cat_summary.loc['Aggressive','avg_dd_sens']:>14.1f} | {'N/A':>14}")
print(f"{'Avg Volatility Sensitivity':<30} | {cat_summary.loc['Conservative','avg_vol_sens']:>14.1f} | {cat_summary.loc['Balanced','avg_vol_sens']:>14.1f} | {cat_summary.loc['Aggressive','avg_vol_sens']:>14.1f} | {'N/A':>14}")
print(f"{'Avg Concentration Pref':<30} | {cat_summary.loc['Conservative','avg_conc_pref']:>14.1f} | {cat_summary.loc['Balanced','avg_conc_pref']:>14.1f} | {cat_summary.loc['Aggressive','avg_conc_pref']:>14.1f} | {'N/A':>14}")
print('-' * 110)

# Performance
print(f"{'Annualized Return':<30} | {cat_summary.loc['Conservative','avg_return']*100:>13.2f}% | {cat_summary.loc['Balanced','avg_return']*100:>13.2f}% | {cat_summary.loc['Aggressive','avg_return']*100:>13.2f}% | {spy_ann_ret*100:>13.2f}%")
print(f"{'Annualized Volatility':<30} | {cat_summary.loc['Conservative','avg_vol']*100:>13.2f}% | {cat_summary.loc['Balanced','avg_vol']*100:>13.2f}% | {cat_summary.loc['Aggressive','avg_vol']*100:>13.2f}% | {spy_ann_vol*100:>13.2f}%")
print(f"{'Sharpe Ratio':<30} | {cat_summary.loc['Conservative','avg_sharpe']:>14.2f} | {cat_summary.loc['Balanced','avg_sharpe']:>14.2f} | {cat_summary.loc['Aggressive','avg_sharpe']:>14.2f} | {spy_sharpe:>14.2f}")
print(f"{'Max Drawdown':<30} | {cat_summary.loc['Conservative','avg_mdd']*100:>13.2f}% | {cat_summary.loc['Balanced','avg_mdd']*100:>13.2f}% | {cat_summary.loc['Aggressive','avg_mdd']*100:>13.2f}% | {spy_mdd*100:>13.2f}%")
print('-' * 110)

# RL Training Penalties
print(f"{'RL TRAINING PENALTIES':^110}")
print('-' * 110)
print(f"{'Avg MDD Penalty':<30} | {cat_summary.loc['Conservative','avg_mdd_penalty']:>14.2f} | {cat_summary.loc['Balanced','avg_mdd_penalty']:>14.2f} | {cat_summary.loc['Aggressive','avg_mdd_penalty']:>14.2f} | {'N/A':>14}")
print(f"{'Avg Volatility Penalty':<30} | {cat_summary.loc['Conservative','avg_vol_penalty']:>14.2f} | {cat_summary.loc['Balanced','avg_vol_penalty']:>14.2f} | {cat_summary.loc['Aggressive','avg_vol_penalty']:>14.2f} | {'N/A':>14}")
print(f"{'Avg Diversity Penalty':<30} | {cat_summary.loc['Conservative','avg_div_penalty']:>14.2f} | {cat_summary.loc['Balanced','avg_div_penalty']:>14.2f} | {cat_summary.loc['Aggressive','avg_div_penalty']:>14.2f} | {'N/A':>14}")
print('-' * 110)

# Portfolio Structure
print(f"{'PORTFOLIO STRUCTURE':^110}")
print('-' * 110)
print(f"{'Avg # Active Assets':<30} | {cat_summary.loc['Conservative','avg_n_assets']:>14.1f} | {cat_summary.loc['Balanced','avg_n_assets']:>14.1f} | {cat_summary.loc['Aggressive','avg_n_assets']:>14.1f} | {'1 (Index)':>14}")
print(f"{'Avg Top-1 Asset Weight':<30} | {cat_summary.loc['Conservative','avg_top1_wt']*100:>13.1f}% | {cat_summary.loc['Balanced','avg_top1_wt']*100:>13.1f}% | {cat_summary.loc['Aggressive','avg_top1_wt']*100:>13.1f}% | {'100.0%':>14}")
print(f"{'Avg Top-5 Asset Weight':<30} | {cat_summary.loc['Conservative','avg_top5_wt']*100:>13.1f}% | {cat_summary.loc['Balanced','avg_top5_wt']*100:>13.1f}% | {cat_summary.loc['Aggressive','avg_top5_wt']*100:>13.1f}% | {'100.0%':>14}")
print('=' * 110)


### 6. Visualization: Risk-Return Scatter & Penalty Heatmap
Visualize how individual profiles cluster in risk-return space and how the RL penalty multipliers scale with user sensitivity inputs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

cat_colors = {'Conservative': '#00B0FF', 'Balanced': '#00E676', 'Aggressive': '#FF3D00'}
cat_markers = {'Conservative': 'o', 'Balanced': 's', 'Aggressive': '^'}

# ── Panel 1: Risk-Return Scatter ──
ax = axes[0]
for _, row in results_df.iterrows():
    ax.scatter(row['ann_vol']*100, row['ann_ret']*100,
              color=cat_colors[row['category']], marker=cat_markers[row['category']],
              s=100, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.scatter(spy_ann_vol*100, spy_ann_ret*100, color='#FFD600', marker='*', s=250, edgecolors='white', linewidth=1, zorder=10, label='S&P 500')

# Add category labels
for cat in ['Conservative', 'Balanced', 'Aggressive']:
    cat_data = results_df[results_df['category'] == cat]
    ax.scatter([], [], color=cat_colors[cat], marker=cat_markers[cat], s=100, label=f'{cat} ({len(cat_data)})')

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk-Return Profile Scatter', fontsize=14, weight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.15)

# ── Panel 2: Penalty Comparison Bar Chart ──
ax = axes[1]
x = np.arange(3)
width = 0.25
cats = ['Conservative', 'Balanced', 'Aggressive']
mdd_vals = [cat_summary.loc[c, 'avg_mdd_penalty'] for c in cats]
vol_vals = [cat_summary.loc[c, 'avg_vol_penalty'] for c in cats]
div_vals = [cat_summary.loc[c, 'avg_div_penalty'] for c in cats]

ax.bar(x - width, mdd_vals, width, label='MDD Penalty', color='#FF5252', alpha=0.85)
ax.bar(x, vol_vals, width, label='Vol Penalty', color='#FF9800', alpha=0.85)
ax.bar(x + width, div_vals, width, label='Div Penalty', color='#7C4DFF', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel('Penalty Magnitude', fontsize=12)
ax.set_title('Avg RL Training Penalties by Category', fontsize=14, weight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.15, axis='y')

# ── Panel 3: Portfolio Concentration ──
ax = axes[2]
for cat in cats:
    cat_data = results_df[results_df['category'] == cat]
    ax.scatter(cat_data['n_assets'], cat_data['top1_weight']*100,
              color=cat_colors[cat], marker=cat_markers[cat], s=100, alpha=0.8,
              edgecolors='white', linewidth=0.5, label=cat)
ax.set_xlabel('# Active Assets', fontsize=12)
ax.set_ylabel('Top-1 Asset Weight (%)', fontsize=12)
ax.set_title('Diversification vs Concentration', fontsize=14, weight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.15)

plt.suptitle(f'TEST_PROFILES Evaluation ({backtest_start_date} → {str(dates[-1].date())})', fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Individual Profile Detail Table ──
print(f'\n{"":>20} INDIVIDUAL PROFILE DETAILS')
print('=' * 130)
print(f'{"Profile Name":<32} | {"Category":<13} | {"Return":>7} | {"Vol":>7} | {"MDD":>7} | {"Sharpe":>6} | {"Assets":>6} | {"Top-1 Wt":>8} | {"Top 5 Holdings"}')
print('-' * 130)
for _, row in results_df.sort_values(['category', 'ann_ret'], ascending=[True, False]).iterrows():
    top5_str = ', '.join([f'{t} ({v:.0%})' for t, v in row['top5'][:3]])
    print(f"{row['name']:<32} | {row['category']:<13} | {row['ann_ret']*100:>6.1f}% | {row['ann_vol']*100:>6.1f}% | {row['mdd']*100:>6.1f}% | {row['sharpe']:>6.2f} | {row['n_assets']:>6} | {row['top1_weight']*100:>7.1f}% | {top5_str}")
print('=' * 130)
